# Notebook 13: Phase 2 results and validation

This final notebook audits the frozen Notebook 08–12 handoffs and synthesizes
their paired comparisons. It does not regenerate physical losses, alter policy
terms, select a new reinsurance objective, or refit the parametric trigger.

**Production:** leave `VERIFY_PROCESSED_ARTIFACTS = True`. The first calculation
cell streams all upstream inventory files through SHA-256, including the large
local ground-motion and loss files. A missing or changed artifact stops the run.

**Metadata preview:** `False` permits inspection using committed metadata only.
It explicitly skips private artifacts, writes to a separate ignored preview
directory, and cannot produce a completed production handoff.

Run all cells in order. Each output is deterministic within the same software
environment. Notebook 08's inventory text uses explicit CRLF reconstruction;
Notebook 08 and 09 handoffs are checked against their published LF byte views.
The frozen inputs are never rewritten. No Git commit, tag,
merge, or release is created by this notebook.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import numpy as np
import pandas as pd
import matplotlib
from IPython.display import display, Image, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "13_phase_2_results_and_validation.ipynb").is_file():
    raise RuntimeError("Start Jupyter from the seismic-correlation-insurance-loss repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
EXPECTED_HELPER_SHA256 = "4f431b143f4d4ed00cc123d7e1c81e4bdde2ac203b60b73b8d83393219819551"
helper_bytes = (ROOT / "tools/phase2_results.py").read_bytes().replace(b"\r\n", b"\n")
if hashlib.sha256(helper_bytes).hexdigest() != EXPECTED_HELPER_SHA256:
    raise RuntimeError("Notebook 13 helper source does not match the reviewed version. Restart after pulling the complete source package.")
from tools.phase2_results import (
    CASE_PREFIXES, DIRECTORIES, HANDOFF_HASHES, PHASE1_COMMIT, OUTPUT_NAME,
    LIMITATIONS, audit_upstream, load_tables, build_results, validate_results,
    make_figures, write_report, write_csv, write_json, inventory_item,
    metadata_path, sha256_file,
)

VERIFY_PROCESSED_ARTIFACTS = True
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 12)
print("Notebook 13 setup ready. Production artifact audit:", VERIFY_PROCESSED_ARTIFACTS)

In [ ]:
# No output is written until the frozen input audit passes.
audit, upstream_handoffs, upstream_validation = audit_upstream(
    ROOT, verify_processed=VERIFY_PROCESSED_ARTIFACTS
)
RUN_IS_PRODUCTION = VERIFY_PROCESSED_ARTIFACTS and audit.status.eq("PASS").all()
OUTPUT_BASE = ROOT / (
    "data/metadata/phase_2/" + OUTPUT_NAME if RUN_IS_PRODUCTION else
    "data/processed/phase_2/" + OUTPUT_NAME + "_preview"
)
FIGURES = ROOT / "data/processed/phase_2" / OUTPUT_NAME / "plots" if RUN_IS_PRODUCTION else OUTPUT_BASE / "plots"
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
output_paths = []
def save_table(name, frame):
    path = OUTPUT_BASE / ("notebook_13_" + name + ".csv")
    write_csv(path, frame)
    output_paths.append(path)
save_table("input_audit", audit)
save_table("upstream_validation", upstream_validation)
print("=" * 78)
print("NOTEBOOK 13 CELL 1: FROZEN UPSTREAM ARTIFACTS AUDITED")
print("=" * 78)
print("Upstream notebooks:          ", len(upstream_handoffs))
print("Artifact checks:             ", len(audit))
print("Passed:                      ", int(audit.status.eq("PASS").sum()))
print("Skipped (preview only):      ", int(audit.status.eq("SKIPPED").sum()))
print("Legacy CRLF byte matches:    ", int(audit.hash_mode.eq("legacy_crlf_reconstruction").sum()))
print("Upstream validation checks:  ", len(upstream_validation))
print("Next: paired executive comparisons and interpretation diagnostics.")

## What the comparison can establish

Insurance and reinsurance use the full two-million-year catalog. Parametric
headlines use the held-out final million years, with the trigger frozen from
I0 training. Keep these periods separate.

The executive table reports amounts and paired changes. Detailed files retain
matching program, design, price, and expense keys. Undefined percentages remain
missing when the baseline is zero. AAL additivity, sparse-loss VaR, attachment
ties, and RAROC assumptions receive explicit diagnostics rather than artificial
rankings. Paired bootstrap intervals are carried forward without resampling.

In [ ]:
tables = load_tables(ROOT)
results = build_results(tables)
for name, frame in results.items():
    save_table(name, frame)
limitations = pd.DataFrame(LIMITATIONS, columns=["limitation_id", "detail"])
save_table("limitations", limitations)
display(results["executive_comparison"])
display(results["attachment_selection_diagnostics"])
display(results["diversification_diagnostics"])
print("NOTEBOOK 13 CELL 2: PAIRED EXECUTIVE COMPARISONS COMPLETE")
print("Result tables:", len(results), "| Paired uncertainty rows:", len(results["paired_uncertainty"]))

In [ ]:
figure_paths = make_figures(tables, results, FIGURES)
output_paths.extend(figure_paths)
for path in figure_paths:
    if path.suffix == ".png":
        display(Image(filename=str(path)))
print("NOTEBOOK 13 CELL 3: FIVE FIGURES COMPLETE (PNG AND SVG)")

In [ ]:
final_validation = validate_results(
    tables, results, audit, upstream_validation,
    verify_processed=VERIFY_PROCESSED_ARTIFACTS,
)
display(final_validation)
critical_failures = final_validation.loc[
    final_validation.severity.eq("critical") & ~final_validation.passed
]
if not critical_failures.empty:
    raise RuntimeError("Notebook 13 synthesis validation failed.")
save_table("final_validation", final_validation)
print("NOTEBOOK 13 CELL 4: SYNTHESIS VALIDATION PASSED")

## Final handoff

The final JSON distinguishes production completion from a metadata preview.
It records frozen upstream hashes, source identity, runtime versions, all output
hashes, and interpretation limitations. The generated report contains the
executive comparisons, attachment identification, and RAROC assumption ranges.

Retain all upstream processed files locally. Publication of the reviewed
metadata, selected figures, and a Phase 2 release remains a separate Git step.

In [ ]:
# Recheck the audit mode at finalization: a preview cannot become a release.
production_complete = bool(
    RUN_IS_PRODUCTION and VERIFY_PROCESSED_ARTIFACTS
    and audit.status.eq("PASS").all() and critical_failures.empty
)
report_path = OUTPUT_BASE / "notebook_13_results_report.md"
write_report(report_path, results, production_complete=production_complete)
output_paths.append(report_path)
specification_path = OUTPUT_BASE / "notebook_13_model_specification.json"
write_json(specification_path, {
    "purpose": "read-only synthesis; no recalibration",
    "money_unit": "2022_USD",
    "full_catalog_years": 2000000,
    "parametric_evaluation_years": [1000001, 2000000],
    "minimum_headline_order_statistic_rank": 20,
    "comparison_baseline": "I0_PHASE1_INDEPENDENT",
    "undefined_percentage_policy": "NaN for zero or missing baseline",
    "legacy_hash_policy": "Notebook 08 inventory text: exact raw or CRLF-reconstructed size/hash; Notebook 08 and 09 handoffs: pinned published LF byte views; all modes explicitly recorded",
    "limitations": dict(LIMITATIONS),
})
output_paths.append(specification_path)
# No stale files from earlier attempts are included; only this run's outputs.
inventory = [inventory_item(ROOT, path) for path in sorted(set(output_paths))]
for item in inventory:
    path = ROOT / item["path"]
    if path.stat().st_size != item["bytes"] or sha256_file(path) != item["sha256"]:
        raise RuntimeError("Output artifact verification failed.")
validation_path = OUTPUT_BASE / "notebook_13_final_validation.csv"
handoff = {
    "notebook13_complete": production_complete,
    "production_artifact_audit_complete": production_complete,
    "release_published": False,
    "mode": "production" if production_complete else "metadata_preview",
    "dependence_cases": list(CASE_PREFIXES),
    "frozen_controls": {"phase1_release": "v1.0.0", "phase1_commit": PHASE1_COMMIT,
                        "catalog_years": 2000000, "occurrences": 10630, "sites": 470},
    "upstream": [{"notebook": n, "path": metadata_path(n, "final_handoff.json"), "sha256": sha}
                 for n, sha in HANDOFF_HASHES.items()],
    "source": {"helper_path": "tools/phase2_results.py", "helper_canonical_lf_sha256": EXPECTED_HELPER_SHA256,
               "notebook_cell_sources_sha256": hashlib.sha256(json.dumps(
                   [{"cell_type": c["cell_type"], "source": "".join(c["source"])}
                    for c in json.loads((ROOT / "13_phase_2_results_and_validation.ipynb").read_text(encoding="utf-8"))["cells"]],
                   sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()},
    "runtime": {"python": platform.python_version(), "numpy": np.__version__,
                "pandas": pd.__version__, "matplotlib": matplotlib.__version__},
    "validation": {"path": validation_path.relative_to(ROOT).as_posix(),
                   "sha256": sha256_file(validation_path), "checks": len(final_validation),
                   "critical_failures": len(critical_failures),
                   "warnings": int((final_validation.severity.eq("warning") & ~final_validation.passed).sum()),
                   "upstream_warning_count": int((upstream_validation.severity.eq("warning") & ~upstream_validation.passed).sum())},
    "artifact_inventory": inventory,
    "next_step": "Review production metadata and figures for Phase 2 release; then project walkthrough and interview preparation.",
}
handoff_path = OUTPUT_BASE / ("notebook_13_final_handoff.json" if production_complete else "notebook_13_preview_handoff.json")
write_json(handoff_path, handoff)
print("=" * 78)
print("NOTEBOOK 13 COMPLETE: PHASE 2 RESULTS AND VALIDATION" if production_complete else
      "NOTEBOOK 13 METADATA PREVIEW COMPLETE: PRODUCTION AUDIT PENDING")
print("=" * 78)
print("Dependence cases:            ", len(CASE_PREFIXES))
print("Upstream artifact checks:    ", len(audit))
print("Executive comparison rows:   ", len(results["executive_comparison"]))
print("Paired uncertainty rows:     ", len(results["paired_uncertainty"]))
print("Figures (PNG and SVG):       ", len(figure_paths) // 2)
print("Validation checks:           ", len(final_validation))
print("Critical failures:           ", len(critical_failures))
print("Final handoff:               ", handoff_path.relative_to(ROOT).as_posix())